In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import urlparse
from webdriver_manager.chrome import ChromeDriverManager
import re
import time

BASE_LISTING_URL = "https://www.noon.com/egypt-en/electronics-and-mobiles/computers-and-accessories/computers-new/laptops/?page={}"
MAX_PAGES = 40  # production mode


def normalize_product_url(url: str) -> str:
    """Keep canonical Noon product URL and remove query tracking params."""
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"


def is_valid_product_url(url: str) -> bool:
    """Accept only real Noon product pages; reject tracking/redirect links."""
    if not url:
        return False

    parsed = urlparse(url)
    host = parsed.netloc.lower()
    path = parsed.path.lower()

    if "tracking.noon.com" in host:
        return False

    if "noon.com" not in host:
        return False

    # Product pages usually contain /p/ in Noon URLs
    if "/p/" not in path:
        return False

    return True


options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1400,1000")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)

all_links = set()

for page in range(1, MAX_PAGES + 1):
    url = BASE_LISTING_URL.format(page)
    driver.get(url)

    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Progressive scroll for lazy-loaded cards
    last_height = 0
    for _ in range(6):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href]")
    page_links_before = len(all_links)

    for a in anchors:
        href = a.get_attribute("href")
        if is_valid_product_url(href):
            all_links.add(normalize_product_url(href))

    page_added = len(all_links) - page_links_before
    print(f"Page {page:02d}: +{page_added} valid links")

all_links = sorted(all_links)
print(f"Total valid product links: {len(all_links)}")

driver.quit()

Page 01: +50 valid links
Page 02: +50 valid links
Page 03: +49 valid links
Page 04: +50 valid links
Page 05: +50 valid links
Page 06: +50 valid links
Page 07: +50 valid links
Page 08: +49 valid links
Page 09: +50 valid links
Page 10: +49 valid links
Page 11: +50 valid links
Page 12: +50 valid links
Page 13: +50 valid links
Page 14: +50 valid links
Page 15: +50 valid links
Page 16: +50 valid links
Page 17: +48 valid links
Page 18: +50 valid links
Page 19: +50 valid links
Page 20: +50 valid links
Page 21: +50 valid links
Page 22: +50 valid links
Page 23: +50 valid links
Page 24: +50 valid links
Page 25: +50 valid links
Page 26: +50 valid links
Page 27: +50 valid links
Page 28: +50 valid links
Page 29: +50 valid links
Page 30: +50 valid links
Page 31: +44 valid links
Page 32: +45 valid links
Page 33: +50 valid links
Page 34: +50 valid links
Page 35: +50 valid links
Page 36: +50 valid links
Page 37: +16 valid links
Page 38: +0 valid links
Page 39: +0 valid links
Page 40: +0 valid links
Pag

In [9]:
import pandas as pd
import re
import time
import json
from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

MAX_PRODUCTS = 1700  # production mode
RETRY_PER_PRODUCT = 2


def is_valid_product_url(url: str) -> bool:
    if not url:
        return False
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    path = parsed.path.lower()
    return ("noon.com" in host) and ("tracking.noon.com" not in host) and ("/p/" in path)


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_for_parsing(text: str) -> str:
    t = text or ""
    # remove trademark/special chars that break regex patterns
    t = t.replace("™", " ").replace("®", " ").replace("“", " ").replace("”", " ")
    return normalize_space(t)


def to_gb(value: int, unit: str) -> int:
    return int(value * 1024) if unit.upper() == "TB" else int(value)


def parse_ram_storage(text: str):
    """
    RAM: <= 128 GB only
    Storage: largest valid capacity from (GB/TB), excluding RAM values
    """
    ram_gb = None
    storage_gb = None

    # Example matches: 16GB RAM, 32 GB DDR5, RAM 8G
    ram_matches = re.findall(r"(\d{1,3})\s*(?:GB|G)\s*(?:RAM|DDR\d?|MEMORY)", text, flags=re.I)
    ram_prefix_matches = re.findall(r"(?:RAM|MEMORY)\s*[:\-]?\s*(\d{1,3})\s*(?:GB|G)", text, flags=re.I)
    ram_candidates = [int(v) for v in (ram_matches + ram_prefix_matches) if 2 <= int(v) <= 128]

    # Generic capacity values: 512GB, 1TB, 2 TB, etc.
    all_caps = re.findall(r"(\d{1,4})\s*(GB|TB)", text, flags=re.I)
    cap_candidates = [to_gb(int(v), u) for v, u in all_caps]

    if ram_candidates:
        # keep largest realistic RAM candidate in case of duplicates
        ram_gb = max(ram_candidates)

    if cap_candidates:
        # Remove very small sizes and probable RAM duplicates
        filtered_storage = [
            c for c in cap_candidates
            if c >= 64 and (ram_gb is None or c != ram_gb)
        ]
        if filtered_storage:
            storage_gb = max(filtered_storage)

    return ram_gb, storage_gb


def parse_storage_type(text: str) -> str:
    t = text.lower()
    if "nvme" in t:
        return "NVMe SSD"
    if "ssd" in t:
        return "SSD"
    if "hdd" in t:
        return "HDD"
    if "emmc" in t:
        return "eMMC"
    return "Unknown"


def parse_cpu_and_gen(text: str):
    lower = normalize_for_parsing(text).lower()

    # Intel Core iX-12345 pattern
    m = re.search(r"\b(i[3579])\s*[-\s]?([0-9]{4,5}[a-z]{0,2})\b", lower, flags=re.I)
    if m:
        cpu_family = m.group(1).upper()
        cpu_model = m.group(2).upper()
        gen_digits = re.search(r"\d{4,5}", cpu_model)
        generation = gen_digits.group(0) if gen_digits else "Unknown"
        return cpu_family, generation, cpu_model

    # Intel Core 7-240H style
    m = re.search(r"core\s+([3579])\s*[-\s]?([0-9]{3,5}[a-z]{0,2})", lower, flags=re.I)
    if m:
        tier = m.group(1)
        model = m.group(2).upper()
        gen_digits = re.search(r"\d{3,5}", model)
        generation = gen_digits.group(0) if gen_digits else "Unknown"
        return f"Core {tier}", generation, model

    # Intel Core Ultra 7 155H style
    m = re.search(r"core\s+ultra\s+([3579])\s*[-\s]?([0-9]{3,4}[a-z]{0,2})", lower, flags=re.I)
    if m:
        tier = m.group(1)
        model = m.group(2).upper()
        gen_digits = re.search(r"\d{3,4}", model)
        generation = gen_digits.group(0) if gen_digits else "Unknown"
        return f"Core Ultra {tier}", generation, model

    # AMD Ryzen AI 9 HX 370 style
    m = re.search(r"(ryzen\s+ai\s*[3579])\s*([a-z]{0,3}\s*)?([0-9]{3,4}[a-z]{0,3})", lower, flags=re.I)
    if m:
        family = normalize_space(m.group(1)).title()
        model = normalize_space((m.group(2) or "") + m.group(3)).upper()
        gen_digits = re.search(r"\d{3,4}", model)
        generation = gen_digits.group(0) if gen_digits else "Unknown"
        return family, generation, model

    # AMD Ryzen 7 7735HS style
    m = re.search(r"(ryzen\s*[3579])\s*[-\s]?([0-9]{4,5}[a-z]{0,3})", lower, flags=re.I)
    if m:
        family = m.group(1).title()
        model = m.group(2).upper()
        gen_digits = re.search(r"\d{4,5}", model)
        generation = gen_digits.group(0) if gen_digits else "Unknown"
        return family, generation, model

    # Snapdragon X pattern
    m = re.search(r"(snapdragon\s*x)\s*([a-z0-9\-\s]{2,12})", lower, flags=re.I)
    if m:
        fam = normalize_space(m.group(1)).title()
        model = normalize_space(m.group(2)).upper()
        return fam, model, model

    # Apple Silicon
    m = re.search(r"\b(m[1234])\b", lower, flags=re.I)
    if m:
        return m.group(1).upper(), m.group(1).upper(), m.group(1).upper()

    return "Unknown", "Unknown", "Unknown"


def parse_gpu(text: str) -> str:
    cleaned = normalize_for_parsing(text)
    patterns = [
        r"\b(RTX\s?\d{3,4}(?:\s?Ti)?)\b",
        r"\b(GTX\s?\d{3,4}(?:\s?Ti)?)\b",
        r"\b(RX\s?\d{3,4}[A-Z]{0,2})\b",
        r"\b(NVIDIA\s+GeForce\s+RTX\s?\d{3,4}(?:\s?Ti)?)\b",
        r"\b(NVIDIA\s+GeForce\s+MX\s?\d{2,4})\b",
        r"\b(Intel\s+Iris\s+Xe\s+Graphics)\b",
        r"\b(Intel\s+UHD\s+Graphics)\b",
        r"\b(UHD\s+Graphics)\b",
        r"\b(AMD\s+Radeon\s+(?:Graphics|Vega\s?\d{0,2}|\w+))\b",
        r"\b(Radeon\s+(?:Graphics|Vega\s?\d{0,2}|\w+))\b",
        r"\b(Apple\s+\d+-Core\s+GPU)\b",
    ]

    for p in patterns:
        m = re.search(p, cleaned, flags=re.I)
        if m:
            value = normalize_space(m.group(1))
            value = re.sub(r"\s+Series$", "", value, flags=re.I)
            return value

    return "Integrated"


def parse_screen_size(text: str):
    m = re.search(r"\b(1[0-9](?:\.\d)?|[2-9](?:\.\d)?)\s*(?:inch|\"|in\b)", text, flags=re.I)
    if m:
        value = float(m.group(1))
        if 10 <= value <= 20:
            return value

    # fallback: "Screen Size 15.6" or "Display 14.0-inch"
    m = re.search(r"(?:screen\s*size|display)\s*[:\-]?\s*(1[0-9](?:\.\d)?)", text, flags=re.I)
    if m:
        value = float(m.group(1))
        if 10 <= value <= 20:
            return value

    return None


def parse_labeled_specs(text: str):
    """Fallback extraction from explicit specification labels."""
    out = {
        "ram_gb": None,
        "storage_gb": None,
        "cpu_text": "",
        "gpu_text": "",
    }

    ram_patterns = [
        r"RAM\s*Size\s*[:\-]?\s*(\d{1,3})\s*(?:GB|G)",
        r"Memory\s*[:\-]?\s*(\d{1,3})\s*(?:GB|G)",
    ]
    for p in ram_patterns:
        m = re.search(p, text, flags=re.I)
        if m:
            value = int(m.group(1))
            if 2 <= value <= 128:
                out["ram_gb"] = value
                break

    storage_patterns = [
        r"Internal\s*Memory\s*[:\-]?\s*(\d{1,4})\s*(GB|TB)",
        r"Storage\s*[:\-]?\s*(\d{1,4})\s*(GB|TB)",
        r"Hard\s*Disk(?:\s*Capacity)?\s*[:\-]?\s*(\d{1,4})\s*(GB|TB)",
    ]
    for p in storage_patterns:
        m = re.search(p, text, flags=re.I)
        if m:
            gb = to_gb(int(m.group(1)), m.group(2))
            if gb >= 64:
                out["storage_gb"] = gb
                break

    for label in ["Processor Version", "CPU", "Processor"]:
        m = re.search(rf"{label}\s*[:\-]?\s*([^\n\|,]+)", text, flags=re.I)
        if m:
            out["cpu_text"] = normalize_space(m.group(1))
            if len(out["cpu_text"]) >= 4:
                break

    for label in ["GPU Version", "Graphics", "Graphic Card"]:
        m = re.search(rf"{label}\s*[:\-]?\s*([^\n\|,]+)", text, flags=re.I)
        if m:
            out["gpu_text"] = normalize_space(m.group(1))
            if len(out["gpu_text"]) >= 4:
                break

    return out


def extract_clean_description(driver):
    # Prefer product details/spec sections and readable text blocks
    selectors = [
        "[data-qa*='description']",
        "[data-qa*='spec']",
        "section",
        "article",
        "ul li",
        "p",
        "span",
    ]

    pieces = []
    seen = set()

    for sel in selectors:
        elements = driver.find_elements(By.CSS_SELECTOR, sel)
        for el in elements[:160]:
            txt = normalize_space(el.text)
            if len(txt) < 18:
                continue

            t = txt.lower()
            if any(bad in t for bad in ["cookie", "privacy", "terms", "delivery", "returns", "login", "sign in", "cash on delivery"]):
                continue

            if txt not in seen:
                pieces.append(txt)
                seen.add(txt)

            if len(" ".join(pieces)) > 3200:
                break

    return " ".join(pieces)[:3200] if pieces else ""


def extract_json_ld_product_data(driver):
    name = ""
    description = ""
    image = ""

    scripts = driver.find_elements(By.CSS_SELECTOR, "script[type='application/ld+json']")
    for sc in scripts:
        raw = (sc.get_attribute("innerHTML") or "").strip()
        if not raw:
            continue

        try:
            payload = json.loads(raw)
        except Exception:
            continue

        objects = payload if isinstance(payload, list) else [payload]

        for obj in objects:
            if not isinstance(obj, dict):
                continue

            typ = str(obj.get("@type", "")).lower()
            if "product" not in typ:
                continue

            if not name:
                name = normalize_space(str(obj.get("name", "")))
            if not description:
                description = normalize_space(str(obj.get("description", "")))

            img_val = obj.get("image", "")
            if isinstance(img_val, list) and img_val:
                image = str(img_val[0])
            elif isinstance(img_val, str):
                image = img_val

            if name or description or image:
                return name, description, image

    return name, description, image


def extract_product_code_from_link(link: str) -> str:
    m = re.search(r"/([A-Z0-9]{8,12})/p/?$", link or "", flags=re.I)
    return m.group(1).upper() if m else ""


def valid_image_url(url: str, product_code: str = "") -> bool:
    if not url:
        return False

    u = url.strip()
    ul = u.lower()

    if not (ul.startswith("http://") or ul.startswith("https://")):
        return False

    # Accept only Noon CDN-like image hosts
    if "f.nooncdn.com" not in ul and "cdn.noonassets.com" not in ul:
        return False

    # Reject CMS/static marketing assets (not product gallery images)
    if "/mpcms/" in ul or "/assets/" in ul:
        return False

    # Remove placeholders/tracking/icons/logo assets
    blacklist = [
        "placeholder", "spacer", "sprite", "icon", "logo", "tracking",
        "pixel", "badge", "avatar", "banner", "gif;base64", "1x1", "thumbnail"
    ]
    if any(k in ul for k in blacklist):
        return False

    if not any(ext in ul for ext in [".jpg", ".jpeg", ".png", ".webp"]):
        return False

    # Strong matching: product image URL should contain the same Noon product code
    if product_code and product_code.lower() not in ul:
        return False

    return True


def parse_srcset_urls(srcset: str):
    urls = []
    for token in (srcset or "").split(","):
        maybe_url = token.strip().split(" ")[0]
        if maybe_url:
            urls.append(maybe_url)
    return urls


def extract_best_product_image(driver, product_code: str = ""):
    priority_selectors = [
        "[data-qa*='gallery'] img",
        "[class*='gallery'] img",
        "[class*='carousel'] img",
        "[class*='product'] img",
        "picture img",
        "img",
    ]

    strict_candidates = []
    fallback_candidates = []

    for sel in priority_selectors:
        imgs = driver.find_elements(By.CSS_SELECTOR, sel)
        for img in imgs[:80]:
            width_attr = img.get_attribute("width") or ""
            height_attr = img.get_attribute("height") or ""

            try:
                w = int(width_attr) if width_attr.isdigit() else 0
                h = int(height_attr) if height_attr.isdigit() else 0
            except Exception:
                w, h = 0, 0

            # Skip likely tiny icons/placeholders
            if (w and w < 120) or (h and h < 120):
                continue

            possible = [
                img.get_attribute("data-zoom-image") or "",
                img.get_attribute("data-src") or "",
                img.get_attribute("src") or "",
            ]
            possible.extend(parse_srcset_urls(img.get_attribute("srcset") or ""))

            for url in possible:
                if valid_image_url(url, product_code=product_code):
                    strict_candidates.append(url)
                elif valid_image_url(url, product_code=""):
                    fallback_candidates.append(url)

        if strict_candidates:
            break

    candidates = strict_candidates if strict_candidates else fallback_candidates

    ranked = sorted(
        set(candidates),
        key=lambda x: (
            "f.nooncdn.com" not in x and "cdn.noonassets.com" not in x,
            "_1." not in x,
            -len(x)
        )
    )

    return ranked[0] if ranked else "N/A"


options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1400,1000")
# Uncomment for faster non-visual scraping:
# options.add_argument("--headless=new")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 12)

records = []
seen_product_codes = set()
seen_fingerprints = set()
links_to_scrape = [u for u in all_links if is_valid_product_url(u)][:MAX_PRODUCTS]

for idx, link in enumerate(links_to_scrape, start=1):
    loaded = False

    for attempt in range(RETRY_PER_PRODUCT + 1):
        try:
            driver.get(link)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
            loaded = True
            break
        except Exception:
            if attempt < RETRY_PER_PRODUCT:
                time.sleep(1)

    if not loaded:
        print(f"[{idx}/{len(links_to_scrape)}] failed: {link}")
        continue

    name = "N/A"
    try:
        name = normalize_space(driver.find_element(By.TAG_NAME, "h1").text)
    except Exception:
        pass

    product_code = extract_product_code_from_link(link)

    # Skip duplicate products early by Noon product code
    if product_code and product_code in seen_product_codes:
        print(f"[{idx}/{len(links_to_scrape)}] skipped duplicate code: {product_code}")
        continue

    jsonld_name, jsonld_desc, jsonld_image = extract_json_ld_product_data(driver)

    if (name == "N/A" or len(name) < 8) and jsonld_name:
        name = jsonld_name

    description = extract_clean_description(driver)
    if (not description or len(description) < 40) and jsonld_desc:
        description = jsonld_desc

    body_txt = normalize_space(driver.find_element(By.TAG_NAME, "body").text)
    combined_text = normalize_space(f"{name} {description} {body_txt}")

    # Price extraction with fallback patterns
    price_clean = None
    try:
        price_candidates = driver.find_elements(By.CSS_SELECTOR, "[data-qa*='price'], [class*='price']")
        for el in price_candidates:
            txt = normalize_space(el.text)
            values = re.findall(r"\b(\d{1,3}(?:,\d{3})+|\d{4,6})\b", txt)
            for v in values:
                n = int(v.replace(",", ""))
                if 1500 <= n <= 500000:
                    price_clean = n
                    break
            if price_clean:
                break
    except Exception:
        pass

    if price_clean is None:
        body_prices = [int(v.replace(",", "")) for v in re.findall(r"\b(\d{1,3}(?:,\d{3})+|\d{4,6})\b", body_txt)]
        body_prices = [x for x in body_prices if 1500 <= x <= 500000]
        price_clean = min(body_prices) if body_prices else None

    ram_gb, storage_gb = parse_ram_storage(combined_text)
    cpu_family, cpu_generation, cpu_model = parse_cpu_and_gen(combined_text)
    gpu = parse_gpu(combined_text)
    screen_size_inch = parse_screen_size(combined_text)
    storage_type = parse_storage_type(combined_text)

    # Fallback from labeled specs blocks
    labeled = parse_labeled_specs(combined_text)
    if ram_gb is None and labeled["ram_gb"] is not None:
        ram_gb = labeled["ram_gb"]
    if storage_gb is None and labeled["storage_gb"] is not None:
        storage_gb = labeled["storage_gb"]

    if cpu_family == "Unknown" and labeled["cpu_text"]:
        cpu_family, cpu_generation, cpu_model = parse_cpu_and_gen(labeled["cpu_text"])

    if gpu == "Integrated" and labeled["gpu_text"]:
        parsed_gpu = parse_gpu(labeled["gpu_text"])
        gpu = parsed_gpu if parsed_gpu else gpu

    image_url = extract_best_product_image(driver, product_code=product_code)
    if image_url == "N/A" and valid_image_url(jsonld_image, product_code=product_code):
        image_url = jsonld_image

    record = {
        "name": name,
        "description": description if description else "N/A",
        "price_egp": price_clean if price_clean is not None else "N/A",
        "image_url": image_url,
        "ram_gb": ram_gb if ram_gb is not None else "N/A",
        "storage_gb": storage_gb if storage_gb is not None else "N/A",
        "storage_type": storage_type,
        "cpu_family": cpu_family,
        "cpu_generation": cpu_generation,
        "cpu_model": cpu_model,
        "gpu": gpu,
        "screen_size_inch": screen_size_inch if screen_size_inch is not None else "N/A",
        "product_code": product_code if product_code else "N/A",
        "link": link,
    }

    # Fallback dedupe in case product code is missing or noisy
    name_key = normalize_space(name).lower()[:120]
    fp = f"{name_key}|{record['price_egp']}|{record['ram_gb']}|{record['storage_gb']}"
    if fp in seen_fingerprints:
        print(f"[{idx}/{len(links_to_scrape)}] skipped duplicate fingerprint")
        continue

    if product_code:
        seen_product_codes.add(product_code)
    seen_fingerprints.add(fp)

    records.append(record)
    print(
        f"[{idx}/{len(links_to_scrape)}] {name[:70]} | "
        f"RAM={record['ram_gb']} | Storage={record['storage_gb']} | "
        f"CPU={record['cpu_family']} {record['cpu_generation']} | GPU={record['gpu']}"
    )


driver.quit()

# Build DataFrame once after scraping (faster + cleaner)
df = pd.DataFrame(records)

# Remove duplicate products by strong keys
if "product_code" in df.columns:
    df = df[df["product_code"].astype(str) != "N/A"]
    df = df.drop_duplicates(subset=["product_code"]).reset_index(drop=True)
else:
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)

# Extra safety dedupe by content fingerprint
df["_fp"] = (
    df["name"].astype(str).str.lower().str[:120] + "|" +
    df["price_egp"].astype(str) + "|" +
    df["ram_gb"].astype(str) + "|" +
    df["storage_gb"].astype(str)
)
df = df.drop_duplicates(subset=["_fp"]).drop(columns=["_fp"]).reset_index(drop=True)

# Optional final cleanup for model-ready dataset
for col in ["name", "description", "image_url", "link", "cpu_family", "cpu_generation", "cpu_model", "gpu", "storage_type"]:
    df[col] = df[col].astype(str).str.strip()

# Save outputs
df.to_csv("laptops_data_clean.csv", index=False, encoding="utf-8-sig")
df.to_json("laptops_data_clean.json", orient="records", force_ascii=False, indent=2)

print("DONE: dataset saved as laptops_data_clean.csv / laptops_data_clean.json")
print("Rows:", len(df))

na_like = ["N/A", "Unknown", "Integrated", ""]
missing_report = {}
for col in ["image_url", "ram_gb", "storage_gb", "cpu_family", "cpu_generation", "gpu", "screen_size_inch", "description"]:
    missing_report[col] = int(df[col].astype(str).isin(na_like).sum())

print("Missing/weak-value report:", missing_report)
print(df.head(5))

[1/1700] 14-Em0006Ne Laptop With 14 Inch FHD /Ryzen 5 -7520U /8GB RAM /512 GB S | RAM=8 | Storage=512 | CPU=Ryzen 5 7520 | GPU=Integrated
[2/1700] 15-3520 Latitude Laptop 15.6 Inch Display Core I7-1165G7 - 8GB RAM 1TB | RAM=8 | Storage=1024 | CPU=Unknown Unknown | GPU=Integrated
[3/1700] 15.6-Inch Display Laptop, Celeron N3060 Processor/4 GB RAM/500 HDD/Int | RAM=4 | Storage=500 | CPU=Unknown Unknown | GPU=Integrated
[4/1700] 15 DC15250 Laptop, 15.6" FHD Touch Display, Intel Core i5-1334U Proces | RAM=16 | Storage=1024 | CPU=I5 1334 | GPU=Intel UHD Graphics
[5/1700] 15 DC15250 Laptop, 15.6" FHD Touch Display, Intel Core i5-1334U Proces | RAM=16 | Storage=512 | CPU=I5 1334 | GPU=Intel UHD Graphics
[6/1700] 15 DC15250 Laptop, 15.6" FHD Touch Display, Intel Core i5-1334U Proces | RAM=8 | Storage=1024 | CPU=I5 1334 | GPU=Intel UHD Graphics
[7/1700] 15 DC15250 Laptop, 15.6" FHD Touch Display, Intel Core i5-1334U Proces | RAM=8 | Storage=512 | CPU=I5 1334 | GPU=Intel UHD Graphics
[8/1700] 15